# Liveness Model Training — Google Colab

Trains the LIVE-vs-SPOOF liveness classifier from the
**Anti-Spoofing Face Liveness Detection for Robust AI-Based Access
Control System** project, using Colab's free GPU.

This notebook mirrors `scripts/train_liveness.py` and
`scripts/preprocess_dataset.py` from the project repo — the same
model code, the same subject-level dataset split (no data leakage),
the same training loop. Train here, download the resulting
`liveness_model.pth`, then copy it into `models/liveness_model.pth`
in your local project folder to use it with the Streamlit app.

**Before running:** obtain a public face anti-spoofing dataset
(MSU-MFSD, CASIA-FASD, Replay-Attack, OULU-NPU, or SiW) under its own
license terms — this notebook does not download or bundle any
dataset. Upload your raw data to Colab (or mount Google Drive) before
Step 2.

**Runtime:** Runtime → Change runtime type → **GPU** (T4 is enough).


## Step 1 — Install dependencies

In [ ]:
!pip install -q opencv-python-headless torch torchvision scikit-learn matplotlib pillow


## Step 2 — Get your dataset onto this Colab instance

Choose ONE of the options below.

**Option A — Google Drive** (recommended for repeated runs):


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Point this at your raw dataset folder in Drive, containing
# live/ and spoof/ subfolders (see the project README's
# "Dataset preparation" section for the expected layout).
RAW_DATASET_DIR = '/content/drive/MyDrive/path/to/your/raw_dataset'


**Option B — direct upload** (small datasets only):

In [ ]:
# from google.colab import files
# uploaded = files.upload()  # upload a .zip of your raw dataset
# !unzip -q -o *.zip -d /content/raw_dataset
# RAW_DATASET_DIR = '/content/raw_dataset'


## Step 3 — Project code

Clone or upload the project repository so this notebook can reuse the
exact same `src/` and `scripts/` modules as the local app (keeping
Colab training and local inference perfectly consistent).


In [ ]:
# If your repo is on GitHub:
# !git clone https://github.com/<your-username>/face-liveness-access-control.git
# %cd face-liveness-access-control

# Otherwise, upload the project folder as a zip and unzip it here,
# then cd into it:
# from google.colab import files
# uploaded = files.upload()
# !unzip -q -o face-liveness-access-control.zip
# %cd face-liveness-access-control

import sys, os
assert os.path.exists('src/face_detection.py'), (
    "Project source not found in the current directory — clone or "
    "upload the repo and cd into it before continuing."
)
sys.path.insert(0, os.getcwd())
print("Project code found.")


## Step 4 — Preprocess the dataset (subject-level split, no data leakage)

In [ ]:
import os
os.environ['RAW_DATASET_DIR'] = RAW_DATASET_DIR


In [ ]:
!python scripts/preprocess_dataset.py \
  --raw-dir "$RAW_DATASET_DIR" \
  --out-dir data \
  --train-ratio 0.7 --val-ratio 0.15 --test-ratio 0.15 \
  --clean


## Step 5 — Train

Same training loop as `scripts/train_liveness.py`: data augmentation,
CrossEntropyLoss + AdamW, best-checkpoint saving by validation
accuracy, early stopping. Colab's GPU makes this much faster than a
laptop CPU.


In [ ]:
!python scripts/train_liveness.py \
  --data-dir data \
  --epochs 15 --batch-size 32 --lr 1e-4 \
  --architecture mobilenet_v3_small \
  --model-out models/liveness_model.pth


## Step 6 — Evaluate

Real, measured metrics on the held-out test set — accuracy, precision,
recall, F1, confusion matrix, and the anti-spoofing-specific
APCER/BPCER/ACER (see the project README for what each metric means).


In [ ]:
!python scripts/evaluate_liveness.py \
  --data-dir data \
  --model-path models/liveness_model.pth


In [ ]:
from IPython.display import Image, display
display(Image(filename='results/confusion_matrix.png'))


## Step 7 — Download the trained model

In [ ]:
from google.colab import files
files.download('models/liveness_model.pth')


## Step 8 — Use it locally

Copy the downloaded `liveness_model.pth` into your local project at:

```
models/liveness_model.pth
```

Then run `streamlit run app.py` locally — the Dashboard tab will
automatically pick it up (no more DEMO MODE banner) and start showing
real LIVE/SPOOF predictions.
